# Next Word Prediction Using LSTM

Adapted from https://github.com/Vishwaaaah/Next_word_prediction_using_LSTM


In [ ]:
import os
import re
import pickle
import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split


def pad_sequences(sequences, maxlen, padding="pre"):
    padded = np.zeros((len(sequences), maxlen), dtype=np.int64)
    for i, seq in enumerate(sequences):
        seq = list(seq)
        if len(seq) > maxlen:
            seq = seq[-maxlen:]
        if not seq:
            continue
        if padding == "pre":
            padded[i, -len(seq) :] = seq
        else:
            padded[i, : len(seq)] = seq
    return padded

In [ ]:
# Retrain the model
# IS_TRAIN_MODE = True

# Load the model from save
IS_TRAIN_MODE = False

In [ ]:
# Load the data

baseDir = os.getcwd()
# curDir = os.path.join(baseDir, "T10 - LLM", "S02 - Recurrent")
curDir = baseDir
filePath = os.path.join(curDir, "hamlet.txt")
print(filePath)

if not os.path.exists(filePath):
    import nltk

    nltk.download("gutenberg")
    from nltk.corpus import gutenberg

    data = gutenberg.raw("shakespeare-hamlet.txt")
    with open(filePath, "w") as file:
        file.write(data)


with open(filePath) as file:
    text = file.read().lower()

In [ ]:
# Show the first 10 lines
for idx, line in enumerate(text.split("\n")):
    print(line)
    if idx > 10:
        break

In [ ]:
# Create a simple tokenizer
class SimpleTokenizer:
    def __init__(self):
        self.word_index = {}
        self.index_word = {}

    def _tokenize(self, text):
        return re.findall(r"\b\w+\b", text.lower())

    def fit_on_texts(self, texts):
        vocab = []
        seen = set()
        for text in texts:
            for token in self._tokenize(text):
                if token not in seen:
                    seen.add(token)
                    vocab.append(token)

        self.word_index = {word: idx + 1 for idx, word in enumerate(vocab)}
        self.index_word = {idx: word for word, idx in self.word_index.items()}

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            tokens = self._tokenize(text)
            seq = [
                self.word_index[token] for token in tokens if token in self.word_index
            ]
            sequences.append(seq)
        return sequences


tokenizer = SimpleTokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
print(total_words)

In [ ]:
# Show the first 20 words as a dictionary
for idx, (k, v) in enumerate(tokenizer.word_index.items()):
    print(f"{k:15s} -> {v:5d}")
    if idx > 20:
        break

In [ ]:
# Test Out of Vocabulary Word
# The tokenizer will ignore the word "CMU" as it is not in the vocabulary
print(tokenizer.texts_to_sequences(["CMU great"]))

In [ ]:
# Creating input-sequence

input_sequences = []
for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[: i + 1]
        input_sequences.append(n_gram_sequence)

input_sequences[:5]

In [ ]:
max_sequence_length = max([len(x) for x in input_sequences])
print(max_sequence_length)

In [ ]:
# Pad sequences
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_length, padding="pre")
)
input_sequences

In [ ]:
# Create predictors and labels
X, yt = input_sequences[:, :-1], input_sequences[:, -1]

In [ ]:
print(X.shape)
print(X[:5])

In [ ]:
yt[:5]

In [ ]:
# Labels for PyTorch CrossEntropyLoss (class indices, not one-hot)
y = yt.astype(np.int64)
print(y.shape)
print(y[:10])

In [ ]:
# Split the data
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Create the model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class NextWordLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=150):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)


model = NextWordLSTM(total_words, embed_dim=100, hidden_dim=150).to(device)


In [ ]:
if IS_TRAIN_MODE:
    dummy_x = torch.randint(0, total_words, (100, max_sequence_length - 1)).to(device)
    dummy_out = model(dummy_x)
    print(dummy_x.shape)
    print(dummy_out.shape)

In [ ]:
from torchinfo import summary

input_size = (100, max_sequence_length - 1)
summary(model, input_size=input_size, dtypes=[torch.long], device=str(device))

In [ ]:
# Train the model

if IS_TRAIN_MODE:
    x_train_t = torch.tensor(x_train, dtype=torch.long)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    x_test_t = torch.tensor(x_test, dtype=torch.long)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    train_ds = torch.utils.data.TensorDataset(x_train_t, y_train_t)
    test_ds = torch.utils.data.TensorDataset(x_test_t, y_test_t)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=256, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=256, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    epochs = 40
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            train_correct += (preds == yb).sum().item()
            train_total += xb.size(0)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
                preds = torch.argmax(logits, dim=1)
                val_correct += (preds == yb).sum().item()
                val_total += xb.size(0)

        train_loss /= max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)
        val_loss /= max(val_total, 1)
        val_acc = val_correct / max(val_total, 1)

        print(
            f"Epoch {epoch + 1:02d}/{epochs} - "
            f"loss: {train_loss:.4f} - acc: {train_acc:.4f} - "
            f"val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}"
        )

    # Save the model
    dateTime = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    model_path = os.path.join(curDir, f"model-{dateTime}.pt")
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "vocab_size": total_words,
            "max_sequence_length": max_sequence_length,
        },
        model_path,
    )
    print(f"Saved model to: {model_path}")

    # Save tokenizer
    with open(os.path.join(curDir, "tokenizer.pkl"), "wb") as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Load the model
if not IS_TRAIN_MODE:
    checkpoint_path = os.path.join(curDir, "lstm-model-20260311-032022.pt")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model = NextWordLSTM(
        vocab_size=checkpoint.get("vocab_size", total_words),
        embed_dim=100,
        hidden_dim=150,
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    max_sequence_length = checkpoint.get("max_sequence_length", max_sequence_length)
    print(model)

In [ ]:
# Function to predict next word


def predict_next_word(model, tokenizer, text, max_sequence_length):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_length:
        token_list = token_list[-max_sequence_length:]

    token_list = pad_sequences(
        [token_list], maxlen=max_sequence_length - 1, padding="pre"
    )

    input_tensor = torch.tensor(token_list, dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)
        predicted_word_index = int(torch.argmax(logits, dim=1).item())

    return tokenizer.index_word.get(predicted_word_index)

In [ ]:
# Predict next word

input_text = "With mirth the king"
print(f"Input text: {input_text}")

token_list = tokenizer.texts_to_sequences([input_text])[0]
print(f"Padded token list: {token_list}")

token_list = pad_sequences([token_list], maxlen=max_sequence_length - 1, padding="pre")
print(f"Token list: {token_list}")

input_tensor = torch.tensor(token_list, dtype=torch.long, device=device)
model.eval()
with torch.no_grad():
    logits = model(input_tensor)
    probs = torch.softmax(logits, dim=1).cpu().numpy()

print(f"Predicted probabilities shape: {probs.shape}")

predicted_word_index = int(np.argmax(probs, axis=1)[0])
print(f"Predicted word index: {predicted_word_index}")

predicted_word = tokenizer.index_word.get(predicted_word_index)
print(f"Predicted next word: {predicted_word}")

In [ ]:
# Generate text
input_text = "I am"

textStr = input_text
print(textStr, end=" ")
for i in range(1, 150):
    next_word = predict_next_word(model, tokenizer, textStr, max_sequence_length)
    if next_word is None:
        break
    print(next_word, end=" ")
    if i % 20 == 0:
        print("\n", end="")
    textStr = textStr + " " + next_word